In [1]:
from dotenv import load_dotenv
load_dotenv('env')

True

In [70]:
import os
import re
import torch
from transformers import AutoTokenizer
from datasets import load_dataset
from tqdm.auto import tqdm

from io import StringIO
from contextlib import redirect_stdout
from time import perf_counter
from huggingface_hub import HfFileSystem, hf_hub_download
import jsonlines
from vllm import LLM, SamplingParams
import pandas as pd
from typing import Tuple, Optional

tqdm.pandas()

In [3]:
BASE_MODEL_ID = "google/gemma-7b"
MODEL_ID = "Hrushi/AIMO-SFT-DDP-Merged-Gemma-7B-N141128-E1-R128-A64"
DATASET_ID = "Hrushi/SFT-MetaMath-40K-TrainingData-Part1"
SAVE_FPATH = "AIMO_Finetuned_Models/AIMO-SFT-DDP-gemma-7b-N141128-E1-R128-A64"


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

In [5]:
%%time
llm = LLM(
    model=SAVE_FPATH,
    tokenizer=SAVE_FPATH,
    # kv_cache_dtype='fp8',
    dtype='bfloat16',
    gpu_memory_utilization=0.95,
    tensor_parallel_size=1,
    max_model_len=1024,
    # trust_remote_code=True,
    # adapter_name_or_path=LORA_ADAPTER_ID,
    swap_space=48,
)

WARNING 06-07 10:48:04 config.py:427] Possibly too large swap space. 48.00 GiB out of the 83.48 GiB total CPU memory is allocated for the swap space.
INFO 06-07 10:48:04 llm_engine.py:161] Initializing an LLM engine (v0.4.3) with config: model='AIMO_Finetuned_Models/AIMO-SFT-DDP-gemma-7b-N141128-E1-R128-A64', speculative_config=None, tokenizer='AIMO_Finetuned_Models/AIMO-SFT-DDP-gemma-7b-N141128-E1-R128-A64', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, rope_scaling=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), seed=0, served_model_name=AIMO_Finetuned_Models/AIMO-SFT-DDP-gemma-7b-N141128-E1-R128-A64)
INFO 06-07 10:48:11 model_runner.py:146]

In [6]:
MATH_ds = load_dataset('hendrycks/competition_math', trust_remote_code=True)['test']
MATH_df = MATH_ds.to_pandas()

In [7]:
GSMK_ds = load_dataset('openai/gsm8k', 'main')['train']
GSMK_df = GSMK_ds.to_pandas()

In [8]:
def get_final_solution(x):
    sols = re.findall(r'\\boxed\{(.*?)\}', x, re.DOTALL)

    if len(sols) == 0:
        sols = re.findall(r'\\boxed (.*?)\$', x, re.DOTALL)
    
    # Check if all are same
    try:
        if all(sol == sols[0] for sol in sols):
            return sols[0]
        else:
            return sols[-1]
    except:
        print(sols)
        print(x)
        raise


In [9]:
MATH_df['final_answer'] = MATH_df['solution'].progress_apply(get_final_solution)
MATH_df['final_answer_is_numeric'] = MATH_df['final_answer'].progress_apply(lambda x: x.isnumeric())

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

In [10]:
MATH_numeric_df = MATH_df[MATH_df['final_answer_is_numeric']].copy(deep=True).reset_index(drop=True)
MATH_numeric_df

,problem,level,type,solution,final_answer,final_answer_is_numeric
0,How many vertical asymptotes does the graph of...,Level 3,Algebra,The denominator of the rational function facto...,2,True
1,What is the positive difference between $120\%...,Level 1,Algebra,One hundred twenty percent of 30 is $120\cdot3...,10,True
2,"If $2^8=4^x$, what is the value of $x$?",Level 1,Algebra,Rewrite $4$ as $2^2$ to find $4^x=2^{2x}$. Si...,4,True
3,What is the 100th term of the arithmetic seque...,Level 2,Algebra,"The common difference is $10 - 6 = 4$, so the ...",402,True
4,Mr. Madoff invests 1000 dollars in a fund that...,Level 4,Algebra,Let $r$ be the annual interest rate. Then aft...,7,True
...,...,...,...,...,...,...
2878,Given $\|\mathbf{v}\| = 5$ and $\|\mathbf{w}\|...,Level 3,Precalculus,Note that\n\begin{align*}\n\operatorname{proj}...,5,True
2879,If $0^\circ < x < 180^\circ$ and $\cos x + \si...,Level 5,Precalculus,"From the given equation, $\cos x = \frac{1}{2}...",14,True
2880,"Let $x_1,$ $x_2,$ $x_3,$ $y_1,$ $y_2,$ and $y_...",Level 5,Precalculus,"In general,\n\[\frac{1}{2} \begin{vmatrix} x_1...",144,True
2881,Compute\n\[\frac{1}{\cos^2 10^\circ} + \frac{1...,Level 4,Precalculus,We can write\n\begin{align*}\n\frac{1}{\cos^2 ...,12,True


In [11]:
system_prompt = """Your are a high school student appearing for your math exam.
You will be given a math problem to solve, and your job is to write the detailed step by step solution, with good mathematical reasoning and using python code for calculations, simplifications, solving equations, etc.

Format help and instructions:
- Problem has been given enclosed in <math_problem></math_problem> tags.
- Every step must be written within <start_of_step><end_of_step> tags.
- Code needs to be always written inside <code_block></code_block> tags - No backticks or triple backticks.
- Code output will be given in <code_output_block></code_output_block> tags

Every calculation should be performed using writing supporting python code."""

In [12]:
idx = 0
problem = MATH_numeric_df['problem'][idx]

messages = [
    {'role': 'system', 'content': system_prompt},
    {'role': 'problem', 'content': problem}
]

print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True).replace('<bos>', ''))

Your are a high school student appearing for your math exam.
You will be given a math problem to solve, and your job is to write the detailed step by step solution, with good mathematical reasoning and using python code for calculations, simplifications, solving equations, etc.

Format help and instructions:
- Problem has been given enclosed in <math_problem></math_problem> tags.
- Every step must be written within <start_of_step><end_of_step> tags.
- Code needs to be always written inside <code_block></code_block> tags - No backticks or triple backticks.
- Code output will be given in <code_output_block></code_output_block> tags

Every calculation should be performed using writing supporting python code.

<math_problem>How many vertical asymptotes does the graph of $y=\frac{2}{x^2+x-6}$ have?</math_problem>
<start_of_solution>



In [13]:
import signal
import functools
from time import sleep

class TimeoutExpired(Exception):
  """Custom exception for timeout"""
  pass

def timeout(seconds=10, error_message='Timeout'):
  """Decorator to timeout a function after a certain number of seconds.

  Args:
      seconds: The maximum number of seconds allowed for the function to run.
      error_message: The message to raise in case of timeout.

  Returns:
      The result of the decorated function if it finishes within the timeout,
      otherwise raises TimeoutExpired with the provided error message.
  """

  def decorator(func):
    @functools.wraps(func)  # Preserve function metadata
    def wrapper(*args, **kwargs):
      def _handle_timeout(signum, frame):
        raise TimeoutExpired(error_message)

      # Set the signal handler and a timer
      signal.signal(signal.SIGALRM, _handle_timeout)
      signal.alarm(seconds)

      try:
        result = func(*args, **kwargs)
      finally:
        # Cancel the alarm if the function finishes before timeout
        signal.alarm(0)
      return result
    return wrapper
  return decorator

In [14]:
@timeout(seconds=60, error_message="Code took too long!")
def run_code(code:str) -> str:
    with redirect_stdout(StringIO()) as f:
        exec(code)
    return f.getvalue().strip()

In [15]:
def get_code(output_str:str) -> str:
    return re.search(r"<code_block>(.*?)</code_block>", output_str, re.DOTALL).group(1).strip()

In [16]:
def get_final_answer(output_str:str) -> str:
    return re.search(r"<final_answer>(.*?)</final_answer>", output_str, re.DOTALL).group(1).strip()

In [17]:
def post_processing(raw_output:str, input_text:str):
    
    if '<final_answer>' in raw_output:
        final_answer = get_final_answer(raw_output)
    else:
        final_answer = None
    
    if '<code_block>' in raw_output:
        
        # Get code block
        code = get_code(raw_output)
        
        # Runt the code
        code_output = run_code(code)
        
        input_text += f"{raw_output}{code_output}</code_output_block>\n"
    else:
        input_text += raw_output
    
    return input_text, final_answer
    

In [18]:
def solve_math_problem(math_problem:str, generation_config:dict, attempts:int=1, verbose:bool=False) -> list:
    
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'problem', 'content': math_problem}
    ]
    
    final_answers = []
    solutions = []
    for idx in range(attempts):
        start_time = perf_counter()
        input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False).replace('<bos>', '')
        final_answer = None
        if verbose:
            print(f"Attempt {idx+1}: ", end="")

        while final_answer is None:
            input_tokens = tokenizer(input_text, return_tensors='pt').to('cuda')
            output_tokens = model.generate(**input_tokens, generation_config=generation_config, tokenizer=tokenizer)

            raw_output = tokenizer.batch_decode(output_tokens)[0]
            raw_output = raw_output.replace('<bos>', '').replace(input_text, '').replace('<pad>', '')

            input_text, final_answer = post_processing(raw_output, input_text)
    
        if verbose:
            print(f'{final_answer: >4} Time Taken: {(perf_counter() - start_time):.2f}s', end='\n')
        
        try:
            final_answers.append(int(final_answer))
        except ValueError:
            final_answers.append(final_answer)
        solutions.append(input_text)
    
    return final_answers, solutions

In [19]:
idx = 100
problem = MATH_numeric_df['problem'][idx].strip()

print(f"Problem: {problem}")
print(f"Original Answer: {MATH_numeric_df['final_answer'][idx]}", end='\n\n')

# final_answers, solutions = solve_math_problem(problem, generation_config, attempts=5)

Problem: The value of $y$ varies inversely as $\sqrt x$ and when $x=24$, $y=15$. What is $x$ when $y=3$?
Original Answer: 600



In [20]:
# # T4-Auto
# Attempt 1:  600 Time Taken: 64.46s
# Attempt 2:  600 Time Taken: 49.98s
# Attempt 3:  600 Time Taken: 34.13s
# Attempt 4:  600 Time Taken: 40.36s
# Attempt 5:  600 Time Taken: 36.30s

# # T4 - Single GPU
# Attempt 1:  600 Time Taken: 59.80s
# Attempt 2:  600 Time Taken: 40.92s
# Attempt 3:  600 Time Taken: 35.01s
# Attempt 4: 10*sqrt(6) Time Taken: 27.71s
# Attempt 5: 602.2904197744444 Time Taken: 35.04s

# # P100
# Attempt 1:  100 Time Taken: 46.58s
# Attempt 2:  600 Time Taken: 31.83s
# Attempt 3:  600 Time Taken: 31.08s
# Attempt 4:  600 Time Taken: 37.94s
# Attempt 5:  600 Time Taken: 34.71s


In [21]:
from collections import Counter

In [22]:
# Counter(final_answers).most_common(1)[0][0]

In [23]:
MATH_numeric_df.head()

,problem,level,type,solution,final_answer,final_answer_is_numeric
0,How many vertical asymptotes does the graph of...,Level 3,Algebra,The denominator of the rational function facto...,2,True
1,What is the positive difference between $120\%...,Level 1,Algebra,One hundred twenty percent of 30 is $120\cdot3...,10,True
2,"If $2^8=4^x$, what is the value of $x$?",Level 1,Algebra,Rewrite $4$ as $2^2$ to find $4^x=2^{2x}$. Si...,4,True
3,What is the 100th term of the arithmetic seque...,Level 2,Algebra,"The common difference is $10 - 6 = 4$, so the ...",402,True
4,Mr. Madoff invests 1000 dollars in a fund that...,Level 4,Algebra,Let $r$ be the annual interest rate. Then aft...,7,True


In [24]:
# fname = "AIMO-SFT-MATH-Eval.jsonl"
# for idx, current_row in tqdm(MATH_numeric_df.iterrows(), total=len(MATH_numeric_df)):
    
#     problem = current_row.problem
    
#     final_answers, solutions = solve_math_problem(problem, generation_config, attempts=5)
    
#     data = {
#         "problem": problem,
#         "level": current_row.level,
#         "type": current_row.type,
#         "solution": current_row.solution,
#         "OFA": current_row.final_answer,
#         "Sols": solutions,
#         "FAs": final_answers,
#         "FA": Counter(final_answers).most_common(1)[0][0]
#     }
    
#     with jsonlines.open(fname, mode='a') as jsonlwriter:
#         jsonlwriter.write(data)
    

In [25]:
math_problem = MATH_numeric_df.problem[0]
messages = [
    {'role': 'system', 'content': system_prompt},
    {'role': 'problem', 'content': math_problem}
]

input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True).replace('<bos>', '')
print(input_text)

Your are a high school student appearing for your math exam.
You will be given a math problem to solve, and your job is to write the detailed step by step solution, with good mathematical reasoning and using python code for calculations, simplifications, solving equations, etc.

Format help and instructions:
- Problem has been given enclosed in <math_problem></math_problem> tags.
- Every step must be written within <start_of_step><end_of_step> tags.
- Code needs to be always written inside <code_block></code_block> tags - No backticks or triple backticks.
- Code output will be given in <code_output_block></code_output_block> tags

Every calculation should be performed using writing supporting python code.

<math_problem>How many vertical asymptotes does the graph of $y=\frac{2}{x^2+x-6}$ have?</math_problem>
<start_of_solution>



In [43]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
sampling_params = SamplingParams(
    n=5,
    temperature=0.2,
    top_p=1,
    top_k=100,
    max_tokens=4096,
    skip_special_tokens=False,
    spaces_between_special_tokens=False,
    stop=['<end_of_step>', '<code_output_block>', '</final_answer>'],
    include_stop_str_in_output=True
)

tokenizer.tokenize('<start_of_solution>')

['<start_of_solution>']

In [27]:
reponse = llm.generate(input_text, sampling_params=sampling_params)

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.22s/it, Generation Speed: 152.40 toks/s]


In [28]:
for out in reponse[0].outputs:
    print(out.text, end="\n\n")

<start_of_step>1
First, we factor the denominator $x^2+x-6$ as $(x+3)(x-2)$.<end_of_step>

<start_of_step>1
First, we factor the denominator: $x^2 + x - 6 = (x+3)(x-2)$.<end_of_step>

<start_of_step>1
The denominator $x^2+x-6$ factors as $(x-2)(x+3)$.<end_of_step>

<start_of_step>1
The denominator of the fraction, $x^2 + x - 6$, can be factored as $(x-2)(x+3)$.<end_of_step>

<start_of_step>1
The denominator of the fraction is $x^2+x-6 = (x-2)(x+3)$.  Since we can't divide by zero, the vertical asymptotes occur where $x-2=0$ or $x+3=0$.  This gives us two vertical asymptotes.<end_of_step>



In [29]:
MATH_numeric_df

,problem,level,type,solution,final_answer,final_answer_is_numeric
0,How many vertical asymptotes does the graph of...,Level 3,Algebra,The denominator of the rational function facto...,2,True
1,What is the positive difference between $120\%...,Level 1,Algebra,One hundred twenty percent of 30 is $120\cdot3...,10,True
2,"If $2^8=4^x$, what is the value of $x$?",Level 1,Algebra,Rewrite $4$ as $2^2$ to find $4^x=2^{2x}$. Si...,4,True
3,What is the 100th term of the arithmetic seque...,Level 2,Algebra,"The common difference is $10 - 6 = 4$, so the ...",402,True
4,Mr. Madoff invests 1000 dollars in a fund that...,Level 4,Algebra,Let $r$ be the annual interest rate. Then aft...,7,True
...,...,...,...,...,...,...
2878,Given $\|\mathbf{v}\| = 5$ and $\|\mathbf{w}\|...,Level 3,Precalculus,Note that\n\begin{align*}\n\operatorname{proj}...,5,True
2879,If $0^\circ < x < 180^\circ$ and $\cos x + \si...,Level 5,Precalculus,"From the given equation, $\cos x = \frac{1}{2}...",14,True
2880,"Let $x_1,$ $x_2,$ $x_3,$ $y_1,$ $y_2,$ and $y_...",Level 5,Precalculus,"In general,\n\[\frac{1}{2} \begin{vmatrix} x_1...",144,True
2881,Compute\n\[\frac{1}{\cos^2 10^\circ} + \frac{1...,Level 4,Precalculus,We can write\n\begin{align*}\n\frac{1}{\cos^2 ...,12,True


In [30]:
MATH_numeric_df['ExpID'] = [f"Exp{idx:0>4}" for idx in range(len(MATH_numeric_df))]
MATH_numeric_df

,problem,level,type,solution,final_answer,final_answer_is_numeric,ExpID
0,How many vertical asymptotes does the graph of...,Level 3,Algebra,The denominator of the rational function facto...,2,True,Exp0000
1,What is the positive difference between $120\%...,Level 1,Algebra,One hundred twenty percent of 30 is $120\cdot3...,10,True,Exp0001
2,"If $2^8=4^x$, what is the value of $x$?",Level 1,Algebra,Rewrite $4$ as $2^2$ to find $4^x=2^{2x}$. Si...,4,True,Exp0002
3,What is the 100th term of the arithmetic seque...,Level 2,Algebra,"The common difference is $10 - 6 = 4$, so the ...",402,True,Exp0003
4,Mr. Madoff invests 1000 dollars in a fund that...,Level 4,Algebra,Let $r$ be the annual interest rate. Then aft...,7,True,Exp0004
...,...,...,...,...,...,...,...
2878,Given $\|\mathbf{v}\| = 5$ and $\|\mathbf{w}\|...,Level 3,Precalculus,Note that\n\begin{align*}\n\operatorname{proj}...,5,True,Exp2878
2879,If $0^\circ < x < 180^\circ$ and $\cos x + \si...,Level 5,Precalculus,"From the given equation, $\cos x = \frac{1}{2}...",14,True,Exp2879
2880,"Let $x_1,$ $x_2,$ $x_3,$ $y_1,$ $y_2,$ and $y_...",Level 5,Precalculus,"In general,\n\[\frac{1}{2} \begin{vmatrix} x_1...",144,True,Exp2880
2881,Compute\n\[\frac{1}{\cos^2 10^\circ} + \frac{1...,Level 4,Precalculus,We can write\n\begin{align*}\n\frac{1}{\cos^2 ...,12,True,Exp2881


In [31]:
all_inputs = []

for prob in tqdm(MATH_numeric_df.problem):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'problem', 'content': prob}
    ]
    
    all_inputs.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True).replace('<bos>', ''))   

  0%|          | 0/2883 [00:00<?, ?it/s]

In [32]:
print(all_inputs[-1])

Your are a high school student appearing for your math exam.
You will be given a math problem to solve, and your job is to write the detailed step by step solution, with good mathematical reasoning and using python code for calculations, simplifications, solving equations, etc.

Format help and instructions:
- Problem has been given enclosed in <math_problem></math_problem> tags.
- Every step must be written within <start_of_step><end_of_step> tags.
- Code needs to be always written inside <code_block></code_block> tags - No backticks or triple backticks.
- Code output will be given in <code_output_block></code_output_block> tags

Every calculation should be performed using writing supporting python code.

<math_problem>Find the smallest positive integer solution to $\tan{19x^{\circ}}=\dfrac{\cos{96^{\circ}}+\sin{96^{\circ}}}{\cos{96^{\circ}}-\sin{96^{\circ}}}$.</math_problem>
<start_of_solution>



In [33]:
all_outputs = llm.generate(
    all_inputs,
    sampling_params,
)

Processed prompts:  18%|█▊        | 514/2883 [01:26<06:41,  5.90it/s, Generation Speed: 2122.64 toks/s]

WARNING 06-07 10:50:19 scheduler.py:1077] Sequence group 607 is preempted by PreemptionMode.SWAP mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=1


Processed prompts:  23%|██▎       | 653/2883 [01:52<07:24,  5.02it/s, Generation Speed: 2161.57 toks/s]

WARNING 06-07 10:50:45 scheduler.py:1077] Sequence group 735 is preempted by PreemptionMode.SWAP mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=51


Processed prompts:  28%|██▊       | 803/2883 [02:24<10:10,  3.40it/s, Generation Speed: 2210.48 toks/s]

...

Processed prompts:  93%|█████████▎| 2687/2883 [10:33<00:21,  9.28it/s, Generation Speed: 2220.75 toks/s]

WARNING 06-07 10:59:26 scheduler.py:1077] Sequence group 2775 is preempted by PreemptionMode.SWAP mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=951


Processed prompts:  97%|█████████▋| 2786/2883 [10:59<00:22,  4.24it/s, Generation Speed: 2216.56 toks/s]

WARNING 06-07 10:59:52 scheduler.py:1077] Sequence group 2852 is preempted by PreemptionMode.SWAP mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=1001


Processed prompts: 100%|██████████| 2883/2883 [11:43<00:00,  4.10it/s, Generation Speed: 2236.00 toks/s]


In [71]:
@timeout(seconds=60, error_message="Code took too long!")
def run_code_v2(code:str) -> Tuple[str, Optional[str], Optional[str]]:
    with redirect_stdout(StringIO()) as f:
        try:
            exec(code)
            error_name = None
            error_msg = None
        except Exception as e:
            error_name = e.__class__.__name__
            error_msg = e
    return f.getvalue().strip(), error_name, error_msg

In [61]:
def get_code(output_str:str) -> str:
    return re.search(r"<code_block>(.*?)</code_block>", output_str, re.DOTALL).group(1).strip()

In [62]:
def get_final_answer(output_str:str) -> str:
    return re.search(r"<final_answer>(.*?)</final_answer>", output_str, re.DOTALL).group(1).strip()

In [67]:
def post_processing_v2(raw_output:str, input_text:str, end_reason:str):
    error_name, error_msg = None, None
    
    if end_reason == '</final_answer>':
        final_answer = get_final_answer(raw_output)
        input_text += raw_output
    else:
        final_answer = None
    
    if end_reason == '<code_output_block>':
        # Get code block
        try:
            code = get_code(raw_output)
            # Runt the code
            code_output, error_name, error_msg = run_code_v2(code)
            input_text += f"{raw_output}{code_output}</code_output_block>\n"
        except AttributeError:
            error_name = "CodeBlockSyntanxError"
            error_msg = None
            input_text += raw_output
        
    elif end_reason == "<end_of_step>":
        input_text += f"{raw_output}\n"
        
    return input_text, final_answer, error_name, error_msg

In [68]:
exp_ids = []
old_prompts = []
responses = []
new_prompts = []
final_answers = []
error_names = []
error_msgs = []

for idx in tqdm(range(len(all_outputs))):
    
    exp_id = MATH_numeric_df.ExpID[idx]
    old_prompt = all_inputs[idx]
    
    for raw_output_details in all_outputs[idx].outputs:
        
        assert old_prompt == all_outputs[idx].prompt
        
        input_text, final_answer, error_name, error_msg = post_processing_v2(raw_output_details.text, all_outputs[idx].prompt, raw_output_details.stop_reason)
        
        exp_ids.append(exp_id)
        old_prompts.append(all_outputs[idx].prompt)
        responses.append(raw_output_details.text)
        new_prompts.append(input_text)
        error_names.append(error_name)
        error_msgs.append(error_msg)
        final_answers.append(final_answer)
        

  0%|          | 0/2883 [00:00<?, ?it/s]

In [72]:
step1_df = pd.DataFrame({
    "ExpID": exp_ids,
    "Prompt1": old_prompts,
    "Response1": responses,
    "Prompt2": new_prompts,
    "Final_Answer": final_answers,
    "Error": error_names,
    "Error_Msg": error_msgs
})
step1_df

,ExpID,Prompt1,Response1,Prompt2,Final_Answer,Error,Error_Msg
0,Exp0000,Your are a high school student appearing for y...,<start_of_step>1\nWe can factor the denominato...,Your are a high school student appearing for y...,None,None,None
1,Exp0000,Your are a high school student appearing for y...,<start_of_step>1\nWe can factor the denominato...,Your are a high school student appearing for y...,None,None,None
2,Exp0000,Your are a high school student appearing for y...,<start_of_step>1\nTo find the vertical asympto...,Your are a high school student appearing for y...,None,None,None
3,Exp0000,Your are a high school student appearing for y...,<start_of_step>1\nThe denominator of the fract...,Your are a high school student appearing for y...,None,None,None
4,Exp0000,Your are a high school student appearing for y...,<start_of_step>1\nTo find the vertical asympto...,Your are a high school student appearing for y...,None,None,None
...,...,...,...,...,...,...,...
14410,Exp2882,Your are a high school student appearing for y...,<start_of_step>1\nWe can rewrite the given equ...,Your are a high school student appearing for y...,None,None,None
14411,Exp2882,Your are a high school student appearing for y...,<start_of_step>1\nWe can simplify the expressi...,Your are a high school student appearing for y...,None,None,None
14412,Exp2882,Your are a high school student appearing for y...,<start_of_step>1\nWe can simplify the expressi...,Your are a high school student appearing for y...,None,None,None
14413,Exp2882,Your are a high school student appearing for y...,<start_of_step>1\nWe can simplify the right si...,Your are a high school student appearing for y...,None,None,None


In [84]:
def process_raw_output(all_outputs:list, main_df:pd.DataFrame) -> pd.DataFrame:
    exp_ids = []
    old_prompts = []
    responses = []
    new_prompts = []
    final_answers = []
    error_names = []
    error_msgs = []

    for idx in tqdm(range(len(all_outputs))):

        exp_id = main_df.ExpID[idx]
        old_prompt = all_inputs[idx]

        for raw_output_details in all_outputs[idx].outputs:

            assert old_prompt == all_outputs[idx].prompt

            input_text, final_answer, error_name, error_msg = post_processing_v2(raw_output_details.text, all_outputs[idx].prompt, raw_output_details.stop_reason)

            exp_ids.append(exp_id)
            old_prompts.append(all_outputs[idx].prompt)
            responses.append(raw_output_details.text)
            new_prompts.append(input_text)
            error_names.append(error_name)
            error_msgs.append(error_msg)
            final_answers.append(final_answer)
    return pd.DataFrame({
        "ExpID": exp_ids,
        "Old_Prompt": old_prompts,
        "Response": responses,
        "Current_Prompt": new_prompts,
        "Final_Answer": final_answers,
        "Error": error_names,
        "Error_Msg": error_msgs
    })
        

In [85]:
test_df = process_raw_output(all_outputs, MATH_numeric_df)


  0%|          | 0/2883 [00:00<?, ?it/s]

,ExpID,Old_Prompt,Response,Current_Prompt,Final_Answer,Error,Error_Msg
0,Exp0000,Your are a high school student appearing for y...,<start_of_step>1\nWe can factor the denominato...,Your are a high school student appearing for y...,None,None,None
1,Exp0000,Your are a high school student appearing for y...,<start_of_step>1\nWe can factor the denominato...,Your are a high school student appearing for y...,None,None,None
2,Exp0000,Your are a high school student appearing for y...,<start_of_step>1\nTo find the vertical asympto...,Your are a high school student appearing for y...,None,None,None
3,Exp0000,Your are a high school student appearing for y...,<start_of_step>1\nThe denominator of the fract...,Your are a high school student appearing for y...,None,None,None
4,Exp0000,Your are a high school student appearing for y...,<start_of_step>1\nTo find the vertical asympto...,Your are a high school student appearing for y...,None,None,None
...,...,...,...,...,...,...,...
14410,Exp2882,Your are a high school student appearing for y...,<start_of_step>1\nWe can rewrite the given equ...,Your are a high school student appearing for y...,None,None,None
14411,Exp2882,Your are a high school student appearing for y...,<start_of_step>1\nWe can simplify the expressi...,Your are a high school student appearing for y...,None,None,None
14412,Exp2882,Your are a high school student appearing for y...,<start_of_step>1\nWe can simplify the expressi...,Your are a high school student appearing for y...,None,None,None
14413,Exp2882,Your are a high school student appearing for y...,<start_of_step>1\nWe can simplify the right si...,Your are a high school student appearing for y...,None,None,None


In [ ]:
df = MATH_numeric_df.copy(deep=True)
problem_col = "problem"
expid_col = "ExpID"
sampling_params = SamplingParams(
    n=5,
    temperature=0.2,
    top_p=1,
    top_k=100,
    max_tokens=1024,
    skip_special_tokens=False,
    spaces_between_special_tokens=False,
    stop=['<end_of_step>', '<code_output_block>', '</final_answer>'],
    include_stop_str_in_output=True
)

final_answers = {}

# First Round
all_inputs = []

for prob in tqdm(df[problem_col]):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'problem', 'content': prob}
    ]

    all_inputs.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True).replace('<bos>', ''))

cal_round = 1
while True:
    
    print(f"\n\n{'==' * 50}\n\nStarting Round {cal_round:0>2}:\t{len(all_inputs)} input prompts to process!\n")
    
    # Perform LLM Calls
    all_outputs = llm.generate(all_inputs, sampling_params)

    # Process the Outputs
    df = process_raw_output(all_outputs=all_outputs, main_df=df)
    df.to_parquet(f"MATH_Inf_Outputs/Round{cal_round:0>2}.parquet", index=False)
    
    # Collect Final Answers if any
    for _, current_row in df[df['Final_Answer'].notna()].iterrows():
        exp_id = current_row['ExpID']

        if exp_id not in final_answers:
            final_answers[exp_id] = [current_row['Final_Answer']]
        else:
            final_answers[exp_id].append(current_row['Final_Answer'])

    # New batch
    new_df = df[(df['Final_Answer'].isna()) & (df['Error'].isna())].copy(deep=True).reset_index(drop=True)
    all_inputs = new_df['Current_Prompt'].values.tolist()
    cal_round += 1
    

  0%|          | 0/2883 [00:00<?, ?it/s]




Starting Round 01:	2883 input prompts to process!



Processed prompts:   8%|▊         | 228/2883 [00:44<06:01,  7.34it/s, Generation Speed: 1680.93 toks/s]

In [86]:
MATH_numeric_df

,problem,level,type,solution,final_answer,final_answer_is_numeric,ExpID
0,How many vertical asymptotes does the graph of...,Level 3,Algebra,The denominator of the rational function facto...,2,True,Exp0000
1,What is the positive difference between $120\%...,Level 1,Algebra,One hundred twenty percent of 30 is $120\cdot3...,10,True,Exp0001
2,"If $2^8=4^x$, what is the value of $x$?",Level 1,Algebra,Rewrite $4$ as $2^2$ to find $4^x=2^{2x}$. Si...,4,True,Exp0002
3,What is the 100th term of the arithmetic seque...,Level 2,Algebra,"The common difference is $10 - 6 = 4$, so the ...",402,True,Exp0003
4,Mr. Madoff invests 1000 dollars in a fund that...,Level 4,Algebra,Let $r$ be the annual interest rate. Then aft...,7,True,Exp0004
...,...,...,...,...,...,...,...
2878,Given $\|\mathbf{v}\| = 5$ and $\|\mathbf{w}\|...,Level 3,Precalculus,Note that\n\begin{align*}\n\operatorname{proj}...,5,True,Exp2878
2879,If $0^\circ < x < 180^\circ$ and $\cos x + \si...,Level 5,Precalculus,"From the given equation, $\cos x = \frac{1}{2}...",14,True,Exp2879
2880,"Let $x_1,$ $x_2,$ $x_3,$ $y_1,$ $y_2,$ and $y_...",Level 5,Precalculus,"In general,\n\[\frac{1}{2} \begin{vmatrix} x_1...",144,True,Exp2880
2881,Compute\n\[\frac{1}{\cos^2 10^\circ} + \frac{1...,Level 4,Precalculus,We can write\n\begin{align*}\n\frac{1}{\cos^2 ...,12,True,Exp2881


In [78]:
step1_df.Error.notna().sum(), step1_df.Final_Answer.notna().sum()

(54, 0)